In [ ]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from typing import Literal
from matplotlib.lines import Line2D

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/pawlo/miniforge3/envs/minimal_dinosaw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
GridType = Literal["grid", "grid_holdout", "random"]
GRID_TYPE:list[GridType] = ["grid", "grid_holdout", "random"]

In [ ]:
selected_model = 'alibi_dv2_coco' # 'dv2' 
model = get_model(selected_model, '../../trained_models', device=DEVICE)
n_dims = 768 if '_b' in selected_model else 384

In [4]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features = []
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    feats = get_features(model, img, device=DEVICE, channel_last=True)
    features.append(feats)

In [5]:
ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'radial', 'diag', 'random',)
ramps_to_results: dict[RampTypes, list[LinearProbeResult]] = {grid_type: {r: [] for r in ramps} for grid_type in GRID_TYPE}

for grid_ in GRID_TYPE:
    match grid_:
        case "random":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = True
        case "grid":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = False
        case "grid_holdout":
            MASK_CUTOFF_FRAC = 0.7
            STEP = 6
            RANDOM_MASK = False
    for ramp in ramps:
        for i in range(n_imgs):
            feats = features[i]
            result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
            ramps_to_results[grid_][ramp].append(result)

In [6]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [7]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none')
    ax.add_collection(pc)

In [ ]:
%%capture
n_rows, n_cols = len(ramps), 8
TITLE_PAD = 20
FS = 20
add_custom_font('resources/fonts', 'Grotesk')
W, H = 3, 2.5

w_spacing = [2 for _ in range(n_cols)]
SPACE_ROW_IDXS = (4,)

FIG_B_COL_OFFSET = 1
FIG_B_W_COLS = 0
FIG_C_COL_OFFSET = FIG_B_COL_OFFSET + FIG_B_W_COLS + 2

fig = plt.figure(figsize=(W * n_cols, H * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, wspace=0.12)
colors: dict[RampTypes, str] = {
    'lr': '#5762D5',
    'ud': '#6370C0',
    'diag': '#6E7DAB',
    'radial': '#575366',
    'random': "#2F2D38",
}
ramp_to_title: dict[RampTypes, str] = {
    'lr': 'Left-right',
    'ud': 'Up-down',
    'diag': 'Diagonal',
    'radial': 'Radial',
    'random': 'Random',
}


top_left_ramp_ax = None

for col, grid_ in enumerate(GRID_TYPE):
    match grid_:
        case "random":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = True
        case "grid":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = False
        case "grid_holdout":
            MASK_CUTOFF_FRAC = 0.7
            STEP = 6
            RANDOM_MASK = False
    for row, ramp in enumerate(ramps):
        h, w = 34, 34
        ramp_arr = get_ramp(ramp, h, w)
        ramp_ax = fig.add_subplot(gs[row, 0 + 2*col])
        ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

        mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        add_red_square_overlay(ramp_ax, mask, 1, 1)

        ramp_ax.set_xticks([])
        ramp_ax.set_yticks([])

        ramp_ax.set_ylabel(ramp_to_title[ramp], fontsize=FS)

        if row == 0:
            ramp_ax.set_title(f'Target ramp:\n{grid_}', fontsize=FS, pad=TITLE_PAD)
            top_left_ramp_ax = ramp_ax

        mean_channel_scores, std_channel_scores, mean_score, _, mean_pred = average_results(ramps_to_results[grid_][ramp])

        mean_pred_ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET + FIG_B_W_COLS + 2*col])
        mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)

        if row == 0:
            mean_pred_ax.set_title('Mean prediction \n(all channels)', fontsize=FS, pad=TITLE_PAD)
        

        mean_pred_ax.set_ylabel(f'$R^{2}:${mean_score:.2f}', fontsize=FS)
        mean_pred_ax.set_xticks([])
        mean_pred_ax.set_yticks([])

pos2 = fig.axes[1].get_position()  # End of column 2
pos3 = fig.axes[10].get_position()

pos4 = fig.axes[11].get_position()  # End of column 4
pos5 = fig.axes[20].get_position()

# positioning line inbetween
x_sep1 = (pos2.x1 + pos3.x0) / 2 -0.003 
x_sep2 = (pos4.x1 + pos5.x0) / 2 -0.003

for x in [x_sep1, x_sep2]:
    fig.add_artist(Line2D(
        [x, x], [0.1, 0.9],
        transform=fig.transFigure,
        color="black",
        linewidth=3
    ))



# plt.tight_layout(pad=0.05)
# plt.savefig('saved/02.png', dpi=300, bbox_inches='tight')
plt.savefig("saved/S1.2.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})